# Retail Sales Performance Analysis
**Project 1 — Python, NumPy & Pandas**

| | |
|---|---|
| **Student** | Khemchand Singh |
| **Dataset** | `data/retail_sales.csv` (502 rows, 10 columns) |
| **Tools** | Python 3.13, NumPy, Pandas |
| **Industry** | Retail / E-commerce |

**Business question:** How is the business performing across products, cities, customers,
payment modes and months — and where should management focus next?

> All numbers below are calculated from the dataset (nothing hard-coded).

## 1. Load the dataset and inspect it

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

df = pd.read_csv(r"C:\Users\Lenovo\Documents\Retail-Sales-Analysis\data\retail_sales.csv")
print("Rows :", len(df))
print("Cols :", len(df.columns))
df.dtypes.to_frame("dtype")

Rows : 502
Cols : 10


,dtype
Order_ID,str
Order_Date,str
City,str
Product,str
Category,str
Quantity,int64
Unit_Price,int64
Customer_ID,str
Payment_Mode,str
Revenue,int64


In [2]:
df.head()

,Order_ID,Order_Date,City,Product,Category,Quantity,Unit_Price,Customer_ID,Payment_Mode,Revenue
0,ORD00000,2025-02-14,Delhi,Headphones,Accessories,5,45999,C0140,UPI,229995
1,ORD00001,2026-01-22,Pune,Headphones,Electronics,1,499,C0121,Cash,499
2,ORD00002,2025-11-24,Hyderabad,Monitor,Electronics,2,45999,C0167,Card,91998
3,ORD00003,2025-08-08,Hyderabad,Webcam,Electronics,3,499,C0172,UPI,1497
4,ORD00004,2025-08-05,Mumbai,Laptop,Accessories,4,499,C0127,UPI,1996


In [3]:
# basic structure
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 502 entries, 0 to 501
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Order_ID      502 non-null    str  
 1   Order_Date    502 non-null    str  
 2   City          490 non-null    str  
 3   Product       502 non-null    str  
 4   Category      502 non-null    str  
 5   Quantity      502 non-null    int64
 6   Unit_Price    502 non-null    int64
 7   Customer_ID   502 non-null    str  
 8   Payment_Mode  502 non-null    str  
 9   Revenue       502 non-null    int64
dtypes: int64(3), str(7)
memory usage: 39.3 KB


## 2. Missing values and duplicate orders

**Observation:** only `City` has missing values (12 rows). Two `Order_ID`s appear twice
(`ORD00010`, `ORD00025`) — exact duplicate records, so they are removed.

In [4]:
missing = df.isna().sum()
print(missing[missing > 0])

dups = df[df.duplicated(subset=["Order_ID"], keep=False)].sort_values("Order_ID")
print("\nDuplicate Order_ID rows:", len(dups))
dups

City    12
dtype: int64

Duplicate Order_ID rows: 4


,Order_ID,Order_Date,City,Product,Category,Quantity,Unit_Price,Customer_ID,Payment_Mode,Revenue
10,ORD00010,2025-09-21,Hyderabad,Mouse,Electronics,3,799,C0067,NetBanking,2397
500,ORD00010,2025-09-21,Hyderabad,Mouse,Electronics,3,799,C0067,NetBanking,2397
25,ORD00025,2025-11-18,Mumbai,Laptop,Electronics,3,799,C0113,UPI,2397
501,ORD00025,2025-11-18,Mumbai,Laptop,Electronics,3,799,C0113,UPI,2397


## 3. Clean the missing `City` values

**Strategy:** missing cities are filled with the **most frequent city inside the same
`Category`** (mode by group), falling back to the global mode. This keeps the city
distribution realistic instead of dropping 12 useful sales rows.

In [5]:
global_mode = df["City"].mode()[0]
cat_mode = df.groupby("Category")["City"].agg(lambda s: s.mode().iat[0] if len(s.mode()) else global_mode)

mask = df["City"].isna()
df.loc[mask, "City"] = df.loc[mask, "Category"].map(cat_mode).fillna(global_mode)
print("filled:", mask.sum(), "rows")

# remove the exact duplicate orders found in step 2
df = df.drop_duplicates(subset=["Order_ID"], keep="first").copy()
print("rows after dedup:", len(df))
df["City"].value_counts()

filled: 12 rows
rows after dedup: 500


City
Bengaluru    113
Delhi        112
Hyderabad    102
Pune         100
Mumbai        73
Name: count, dtype: int64

## 4. Convert `Order_Date` and extract month / weekday

In [6]:
df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce")
df["Month"] = df["Order_Date"].dt.month
df["Month_Name"] = df["Order_Date"].dt.month_name()
df["Weekday"] = df["Order_Date"].dt.day_name()

print("Range:", df["Order_Date"].min().date(), "to", df["Order_Date"].max().date())
df[["Order_ID", "Order_Date", "Month", "Month_Name", "Weekday"]].head()

Range: 2025-01-03 to 2026-05-15


,Order_ID,Order_Date,Month,Month_Name,Weekday
0,ORD00000,2025-02-14,2,February,Friday
1,ORD00001,2026-01-22,1,January,Thursday
2,ORD00002,2025-11-24,11,November,Monday
3,ORD00003,2025-08-08,8,August,Friday
4,ORD00004,2025-08-05,8,August,Tuesday


## 5. Row-wise revenue and verification

`Revenue` should equal `Quantity × Unit_Price`. The check below proves whether the supplied
column is trustworthy.

In [7]:
df["Calc_Revenue"] = df["Quantity"] * df["Unit_Price"]
diff = (df["Calc_Revenue"] - df["Revenue"]).abs()

print("max difference      :", round(diff.max(), 4))
print("rows differing >0.01:", int((diff > 0.01).sum()))
print("=> supplied Revenue matches Quantity x Unit_Price" if diff.max() < 0.01 else "=> mismatch found")

max difference      : 0
rows differing >0.01: 0
=> supplied Revenue matches Quantity x Unit_Price


## 6. Total revenue, total quantity and average order value

In [8]:
summary = pd.DataFrame({
    "Metric": ["Total revenue", "Total quantity", "Unique orders", "Average order value",
               "Median order value", "Largest single order"],
    "Value": [
        df["Revenue"].sum(),
        df["Quantity"].sum(),
        df["Order_ID"].nunique(),
        df["Revenue"].mean(),
        df["Revenue"].median(),
        df["Revenue"].max(),
    ],
})
summary["Value"] = summary["Value"].map(lambda v: f"{v:,.2f}")
summary

,Metric,Value
0,Total revenue,"13,954,120.00"
1,Total quantity,"1,480.00"
2,Unique orders,500.00
3,Average order value,"27,908.24"
4,Median order value,"4,998.00"
5,Largest single order,"229,995.00"


## 7. Top 10 products by revenue

In [9]:
top_products = (df.groupby("Product")
                  .agg(Revenue=("Revenue", "sum"), Units=("Quantity", "sum"), Orders=("Order_ID", "nunique"))
                  .sort_values("Revenue", ascending=False))
top_products["Share_%"] = (top_products["Revenue"] / top_products["Revenue"].sum() * 100).round(1)
top_products.head(10).round(2)

,Revenue,Units,Orders,Share_%
Product,,,,
Keyboard,3438656,244,88,24.6
Headphones,2815469,231,79,20.2
Laptop,2371954,246,81,17.0
Mouse,2370572,228,78,17.0
Webcam,1588503,297,94,11.4
Monitor,1368966,234,80,9.8


## 8. Cities ranked by revenue

In [10]:
city = (df.groupby("City")
          .agg(Revenue=("Revenue", "sum"), Orders=("Order_ID", "nunique"), Units=("Quantity", "sum"))
          .sort_values("Revenue", ascending=False))
city["Avg_Order_Value"] = (city["Revenue"] / city["Orders"]).round(2)
city["Share_%"] = (city["Revenue"] / city["Revenue"].sum() * 100).round(1)
city.round(2)

,Revenue,Orders,Units,Avg_Order_Value,Share_%
City,,,,,
Delhi,3397468,112,332,30334.54,24.3
Bengaluru,3148159,113,341,27859.81,22.6
Pune,2881694,100,306,28816.94,20.7
Hyderabad,2620717,102,283,25693.30,18.8
Mumbai,1906082,73,218,26110.71,13.7


## 9. Monthly revenue trend and the best month

In [11]:
monthly = (df.groupby(["Month", "Month_Name"])["Revenue"]
             .sum().reset_index().sort_values("Month"))
monthly["Share_%"] = (monthly["Revenue"] / monthly["Revenue"].sum() * 100).round(1)
monthly["Revenue"] = monthly["Revenue"].round(2)

best = monthly.loc[monthly["Revenue"].idxmax()]
print(f"Best month: {best['Month_Name']} -> {best['Revenue']:,.2f}")
monthly

Best month: February -> 2,003,009.00


,Month,Month_Name,Revenue,Share_%
0,1,January,1968531,14.1
1,2,February,2003009,14.4
2,3,March,1212209,8.7
3,4,April,1407122,10.1
4,5,May,1179881,8.5
5,6,June,431919,3.1
6,7,July,624525,4.5
7,8,August,1144377,8.2
8,9,September,895804,6.4
9,10,October,1060301,7.6


## 10. Top 10 customers by total spending

In [12]:
customers = (df.groupby("Customer_ID")
               .agg(Revenue=("Revenue", "sum"), Orders=("Order_ID", "nunique"))
               .sort_values("Revenue", ascending=False))
customers["Avg_Order_Value"] = (customers["Revenue"] / customers["Orders"]).round(2)
customers.head(10).round(2)

,Revenue,Orders,Avg_Order_Value
Customer_ID,,,
C0018,445986,4,111496.50
C0043,425985,6,70997.50
C0034,392985,5,78597.00
C0144,376884,4,94221.00
C0083,321993,2,160996.50
C0165,302491,4,75622.75
C0132,278984,4,69746.00
C0150,277986,4,69496.50
C0073,269084,6,44847.33


## 11. Payment methods: orders vs revenue

In [13]:
pay = (df.groupby("Payment_Mode")
         .agg(Orders=("Order_ID", "nunique"), Revenue=("Revenue", "sum"))
         .sort_values("Revenue", ascending=False))
pay["Share_%"] = (pay["Revenue"] / pay["Revenue"].sum() * 100).round(1)
pay["Avg_Order_Value"] = (pay["Revenue"] / pay["Orders"]).round(2)
pay.round(2)

,Orders,Revenue,Share_%,Avg_Order_Value
Payment_Mode,,,,
UPI,134,4246098,30.4,31687.30
Card,110,3601671,25.8,32742.46
Cash,137,3599772,25.8,26275.71
NetBanking,119,2506579,18.0,21063.69


## 12. Unusually large orders (IQR method)

`upper = Q3 + 1.5 × IQR`. Orders above this line are statistically unusual and should be
checked by the sales team (possible bulk / corporate orders).

In [14]:
q1, q3 = np.percentile(df["Revenue"], [25, 75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr

outliers = df[(df["Revenue"] < lower) | (df["Revenue"] > upper)]
print(f"Q1={q1:,.2f}  Q3={q3:,.2f}  IQR={iqr:,.2f}")
print(f"Bounds: {lower:,.2f} .. {upper:,.2f}")
print("Outlier orders:", len(outliers), f"({len(outliers)/len(df)*100:.1f}% of orders)")
print("Outlier revenue share: {:.1f}%".format(outliers["Revenue"].sum() / df["Revenue"].sum() * 100))
outliers.nlargest(8, "Revenue")[["Order_ID", "City", "Product", "Quantity", "Unit_Price", "Revenue"]]

Q1=2,397.00  Q3=38,997.00  IQR=36,600.00
Bounds: -52,503.00 .. 93,897.00
Outlier orders: 32 (6.4% of orders)
Outlier revenue share: 41.5%


,Order_ID,City,Product,Quantity,Unit_Price,Revenue
0,ORD00000,Delhi,Headphones,5,45999,229995
13,ORD00013,Hyderabad,Mouse,5,45999,229995
34,ORD00034,Bengaluru,Laptop,5,45999,229995
97,ORD00097,Pune,Headphones,5,45999,229995
104,ORD00104,Hyderabad,Mouse,5,45999,229995
131,ORD00131,Mumbai,Headphones,5,45999,229995
147,ORD00147,Bengaluru,Headphones,5,45999,229995
158,ORD00158,Delhi,Keyboard,5,45999,229995


## 13. Product-category summary (`groupby`)

In [15]:
cat = (df.groupby("Category")
         .agg(Orders=("Order_ID", "nunique"), Revenue=("Revenue", "sum"), Units=("Quantity", "sum"))
         .sort_values("Revenue", ascending=False))
cat["Share_%"] = (cat["Revenue"] / cat["Revenue"].sum() * 100).round(1)
cat["Avg_Order_Value"] = (cat["Revenue"] / cat["Orders"]).round(2)
cat.round(2)

,Orders,Revenue,Units,Share_%,Avg_Order_Value
Category,,,,,
Accessories,240,7492779,721,53.7,31219.91
Electronics,260,6461341,759,46.3,24851.31


### 13.1 Data-quality check: is `Category` a product attribute?

Before drawing conclusions from the category table, the analyst must confirm that each product
belongs to exactly one category. If a product appears under both categories, `Category` is only a
**transaction-level label**, and category results describe *where revenue was booked*, not the
product line itself.

In [16]:
check = pd.DataFrame({
    "distinct_categories": df.groupby("Product")["Category"].nunique(),
    "categories_seen": df.groupby("Product")["Category"].agg(lambda s: ", ".join(sorted(s.unique()))),
    "distinct_prices": df.groupby("Product")["Unit_Price"].nunique(),
    "min_price": df.groupby("Product")["Unit_Price"].min(),
    "max_price": df.groupby("Product")["Unit_Price"].max(),
})
check

,distinct_categories,categories_seen,distinct_prices,min_price,max_price
Product,,,,,
Headphones,2,"Accessories, Electronics",6,499,45999
Keyboard,2,"Accessories, Electronics",6,499,45999
Laptop,2,"Accessories, Electronics",6,499,45999
Monitor,2,"Accessories, Electronics",6,499,45999
Mouse,2,"Accessories, Electronics",6,499,45999
Webcam,2,"Accessories, Electronics",6,499,45999


In [17]:
inconsistent = df.groupby("Product")["Category"].nunique().gt(1).sum()
print("products mapped to more than one category:", inconsistent, "of", df["Product"].nunique())
if inconsistent:
    print("=> Category is a transaction label, not a fixed product attribute.")
    print("   Category shares are valid for revenue reporting, but must NOT be read as")
    print("   'this product line belongs to this category'.")
print("\nprice spread per product: every product sells between",
      f"{df['Unit_Price'].min():,} and {df['Unit_Price'].max():,}",
      "=> price positioning per product cannot be inferred from this synthetic dataset.")

products mapped to more than one category: 6 of 6
=> Category is a transaction label, not a fixed product attribute.
   Category shares are valid for revenue reporting, but must NOT be read as
   'this product line belongs to this category'.

price spread per product: every product sells between 499 and 45,999 => price positioning per product cannot be inferred from this synthetic dataset.


## 14. NumPy statistics on order revenue

In [18]:
rev = df["Revenue"].to_numpy()

stats = {
    "mean": np.mean(rev),
    "median": np.median(rev),
    "std (sample)": np.std(rev, ddof=1),
    "min": np.min(rev),
    "max": np.max(rev),
    "25th pct": np.percentile(rev, 25),
    "75th pct": np.percentile(rev, 75),
    "90th pct": np.percentile(rev, 90),
}
pd.Series(stats).round(2).to_frame("value")

,value
mean,27908.24
median,4998.00
std (sample),47746.53
min,499.00
max,229995.00
25th pct,2397.00
75th pct,38997.00
90th pct,91998.00


## 15. Weekday pattern

In [19]:
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
wd = (df.groupby("Weekday")
        .agg(Revenue=("Revenue", "sum"), Orders=("Order_ID", "nunique"))
        .reindex(order))
wd["Avg_Order_Value"] = (wd["Revenue"] / wd["Orders"]).round(2)
wd.round(2)

,Revenue,Orders,Avg_Order_Value
Weekday,,,
Monday,2073151,78,26578.86
Tuesday,1790304,70,25575.77
Wednesday,1797998,69,26057.94
Thursday,1441979,73,19753.14
Friday,1802203,65,27726.20
Saturday,2973291,73,40730.01
Sunday,2075194,72,28822.14


## 16. Save the cleaned dataset

The cleaned file is written to `data/retail_sales_clean.csv` so every number above can be
reproduced.

In [20]:
clean_path = r"C:\Users\Lenovo\Documents\Retail-Sales-Analysis\data\retail_sales_clean.csv"
df.drop(columns=["Calc_Revenue"]).to_csv(clean_path, index=False)
print("saved:", clean_path, "| rows:", len(df))

saved: C:\Users\Lenovo\Documents\Retail-Sales-Analysis\data\retail_sales_clean.csv | rows: 500


## 17. Five business recommendations (evidence-based)

1. **Protect the top revenue products.** `Keyboard`, `Headphones` and `Laptop` are the highest
   revenue generators — keep stock availability high and bundle accessories with them.
2. **Focus marketing on Delhi and Bengaluru.** They contribute the largest revenue share, and
   Delhi also has the highest average order value, so premium offers fit there best.
3. **Plan for the February–April peak and June–July dip.** Reallocate ad spend to strong months
   and run clearance/promo campaigns in the low months (June revenue share is only ~3%).
4. **Grow UPI and Card share.** UPI already leads with ~30% revenue share; cash-on-delivery still
   holds ~26% — cashback offers on UPI/Card reduce handling cost and improve prepaid ratio.
5. **Audit the 32 statistically unusual orders.** They represent a disproportionate revenue share;
   confirming they are genuine bulk/corporate orders (not data errors) protects reporting quality,
   and those customers deserve a key-account treatment.

### Limitations
- The dataset is synthetic and covers ~17 months; seasonality conclusions should be re-validated
  with more history.
- Missing `City` values were imputed (12 rows), which slightly smooths city-level results.
- `Category` is **not** a fixed product attribute in this dataset: all six products appear under
  both Accessories and Electronics, and every product sells across the full price range
  (₹499–₹45,999). Category shares are therefore valid for revenue reporting, but must not be read
  as product-line membership or price positioning.
- Correlation between variables is not causal; recommendations are directional, not guaranteed.